# 04 — Fit the production model

The **freeze step**. `02`/`03` fitted a fresh model per fold on seasons `< T`; this notebook fits
**once on all 24 settled seasons (2002–2025)** and pins the result. No holdout — the evaluation is
finished and the folds already answered the question.

**It does not choose the model.** §8 is explicit that the choice is made under the preregistered
rules and merely *executed* here; choosing after seeing fold results would be selection on the
evaluation set. The choice is **M4-c**, forced by two requirements fixed in advance:

1. **§7 gate A** requires the display to show "projected wins **and the distribution**" — M4-c is
   the only model that emits a distribution.
2. **M5 cannot produce a 2026 row** — no 2026 archived line exists and A2.2 makes it fail closed,
   while the structural models remain possible.

**Writes:** `models/win_totals_model.pkl` and `artifacts/model_metadata.json`.
Nothing the live page reads — `05` consumes the pkl; the site reads only `05`'s CSV.

```bash
papermill futures/season_team_totals/04_fit_production.ipynb /tmp/out.ipynb
```

## Section 1 — Parameters

Paths, the simulation size and the seed. The model identity, the folds, the gates and the venue rule
are all read from frozen artifacts.

In [1]:
AUDIT_PATH   = None      # None -> futures/artifacts/data_audit.json
PANEL_PATH   = None      # None -> futures/data/team_season_panel.parquet
META_PATH    = None      # None -> futures/artifacts/dataset_metadata.json
VENUE_PATH   = None      # None -> futures/data/season_schedule_context.parquet
SCHED_PATH   = None      # None -> futures/data/schedules_snapshot.parquet
DIST_PATH    = None      # None -> futures/artifacts/distribution_eval.json  (03)
COMP_PATH    = None      # None -> futures/artifacts/model_comparison.json   (02)
MODEL_PATH   = None      # None -> futures/models/win_totals_model.pkl
OUT_PATH     = None      # None -> futures/artifacts/model_metadata.json
N_SIMS       = 20000
SEED         = 20260802
WRITE_ARTIFACTS = True
RUN_TESTS    = True

### Interpreting the output

Silent. Note the absence of a `MODEL_FAMILY` parameter: the production model cannot be selected by
whoever runs the notebook.

### What these tests guard

That neither the model identity nor any threshold is injectable, and that the simulation size
matches what `03` evaluated — a production artifact simulated at a different resolution than the one
that was calibrated would not be the same object.

In [2]:
if RUN_TESTS:
    assert isinstance(SEED, int) and N_SIMS >= 10000
    for _n in ("MODEL_FAMILY", "TAU", "ALPHA", "GATE_A_THRESHOLD", "FOLDS"):
        assert _n not in dir(), f"{_n} must be read or fitted, never a parameter"
    print(f"✓ Section 1 tests passed | {N_SIMS:,} simulations, seed {SEED}, model identity not injectable")

✓ Section 1 tests passed | 20,000 simulations, seed 20260802, model identity not injectable


### Reading the test result

The parameter surface is paths, a seed and a simulation size. Does **not** prove the upstream
artifacts exist or that the gate passed — Section 2.

## Section 2 — Gate: audit GO, and §7 gate A actually passed

Two conditions, both read from frozen artifacts rather than asserted in prose:

* the audit verdict is `GO` or `GO-TIER-B`, and
* **§7 gate A passed** — ΔMAE vs persistence ≤ −0.15 pooled with ≥60% of folds improving.

Gate A is re-derived here from `03`'s recorded numbers against `02`'s recorded thresholds, so the
production fit cannot proceed on an unverified claim.

In [3]:
import hashlib
import json
import platform
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


def _find_repo_root(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / "app.py").exists() and (p / "futures").is_dir():
            return p
    raise RuntimeError(f"repo root not found above {start}")


REPO    = _find_repo_root(Path.cwd())
FUTURES = REPO / "futures"
ART     = FUTURES / "artifacts"
DATA    = FUTURES / "data"
MODELS  = FUTURES / "models"
AUDIT = Path(AUDIT_PATH) if AUDIT_PATH else ART / "data_audit.json"
PANEL = Path(PANEL_PATH) if PANEL_PATH else DATA / "team_season_panel.parquet"
META  = Path(META_PATH) if META_PATH else ART / "dataset_metadata.json"
VENUE = Path(VENUE_PATH) if VENUE_PATH else DATA / "season_schedule_context.parquet"
SCHED = Path(SCHED_PATH) if SCHED_PATH else DATA / "schedules_snapshot.parquet"
DIST  = Path(DIST_PATH) if DIST_PATH else ART / "distribution_eval.json"
COMP  = Path(COMP_PATH) if COMP_PATH else ART / "model_comparison.json"
MODEL = Path(MODEL_PATH) if MODEL_PATH else MODELS / "win_totals_model.pkl"
OUT   = Path(OUT_PATH) if OUT_PATH else ART / "model_metadata.json"


def _rel(p):
    p = Path(p)
    try:
        return p.resolve().relative_to(REPO).as_posix()
    except ValueError:
        return str(p.resolve())


def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for c in iter(lambda: fh.read(1 << 20), b""):
            h.update(c)
    return h.hexdigest()


def sha256_frame(df):
    return hashlib.sha256(pd.util.hash_pandas_object(df.reset_index(drop=True),
                                                     index=False).values.tobytes()).hexdigest()


audit = json.loads(AUDIT.read_text(encoding="utf-8"))
meta  = json.loads(META.read_text(encoding="utf-8"))
dist  = json.loads(DIST.read_text(encoding="utf-8"))
comp  = json.loads(COMP.read_text(encoding="utf-8"))

VERDICT = audit["verdict"]
if not VERDICT.startswith("GO"):
    raise RuntimeError(f"audit verdict is {VERDICT} — 04 must not run")
TIER_C_OPEN = bool(audit.get("tier_c_open", False))

# §7 gate A, re-derived from 03's recorded numbers against 02's recorded thresholds
GATE_A_DELTA = float(comp["gates"]["gate_A_delta_threshold"])
GATE_WIN     = float(comp["gates"]["fold_win_threshold"])
pe = dist["point_estimate"]
m4_delta_b1, m4_win_b1 = float(pe["delta_vs_B1_headline"]), float(pe["fold_win_vs_B1"])
GATE_A_M4 = bool(m4_delta_b1 <= GATE_A_DELTA and m4_win_b1 >= GATE_WIN)
# M4-c is a calibration variant of the same §6 family; check its margin too
m4c_mae = float(dist["amendment_3"]["M4c"]["headline"]["mae"])
b1_mae  = float(pe["pooled_mae_headline"]["B1_persistence"])
m4c_delta_b1 = m4c_mae - b1_mae
GATE_A_M4C = bool(m4c_delta_b1 <= GATE_A_DELTA)
A3_PASS = bool(dist["amendment_3"]["acceptance_pass"])

if not (GATE_A_M4 and GATE_A_M4C and A3_PASS):
    raise RuntimeError("§7 gate A or A3 acceptance did not pass — 04 must not fit a production model")

print(f"audit verdict     : {VERDICT}   (gate C open: {TIER_C_OPEN})")
print(f"gate A thresholds : dMAE <= {GATE_A_DELTA}, fold win >= {GATE_WIN:.0%}")
print(f"  M4   dMAE vs B1 {m4_delta_b1:+.4f}, folds {m4_win_b1:.0%}  -> gate A {GATE_A_M4}")
print(f"  M4-c dMAE vs B1 {m4c_delta_b1:+.4f}                        -> gate A {GATE_A_M4C}")
print(f"A3 calibration acceptance (coverage80 in band): {A3_PASS}")
print(f"gate B (market): {[v['gate_B'] for v in comp['gates']['verdicts']]} — all False, and 04 does not reopen it")

audit verdict     : GO-TIER-B   (gate C open: False)
gate A thresholds : dMAE <= -0.15, fold win >= 60%
  M4   dMAE vs B1 -0.4751, folds 100%  -> gate A True
  M4-c dMAE vs B1 -0.4646                        -> gate A True
A3 calibration acceptance (coverage80 in band): True
gate B (market): [False, False, False, False] — all False, and 04 does not reopen it


### Interpreting the output

Gate A clears with a wide margin — **−0.475** against a −0.15 threshold on **100%** of folds — and
A3's calibration acceptance passed, so a production fit is licensed.

The last line is the one that must travel with the artifact: **gate B is False for every model.**
Being licensed to ship a descriptive projection is not being licensed to claim it beats the market.

### What these tests guard

That the fit cannot proceed unless the gate genuinely passed — the check raises, it does not warn —
and that gate B's failure is carried forward rather than quietly dropped once a model is being
shipped.

In [4]:
if RUN_TESTS:
    assert VERDICT in ("GO", "GO-TIER-B") and TIER_C_OPEN is False
    assert GATE_A_M4 and GATE_A_M4C and A3_PASS
    assert all(v["gate_B"] is False for v in comp["gates"]["verdicts"]), \
        "gate B is recorded as passing somewhere — 04 must not be the place that discovers it"
    assert dist["amendment_3"]["gate_B_unchanged"] is True
    # the gate is a real barrier: a NO-GO verdict must stop this notebook
    assert not "NO-GO".startswith("GO")
    print(f"✓ Section 2 tests passed | gate A passed ({m4_delta_b1:+.3f} vs {GATE_A_DELTA}), "
          f"A3 accepted, gate B False on all {len(comp['gates']['verdicts'])} models and carried forward")

✓ Section 2 tests passed | gate A passed (-0.475 vs -0.15), A3 accepted, gate B False on all 4 models and carried forward


### Reading the test result

The licence to fit is derived from recorded numbers, not assumed. Does **not** make the projection
accurate — gate A is a low bar (beat persistence), and persistence is a weak baseline.

## Section 3 — Load inputs and prove the engine reproduces `03`

The estimator lives in `m4_engine.py` so `03` and `04` cannot drift apart. Before fitting anything,
this section **re-fits fold 2025's exact training window through the module and checks the fitted
constants against the values `03` recorded** — home field, σ and the tie threshold, to full
precision.

If the module were not the same estimator, that check fails and no artifact is written.

In [5]:
sys.path.insert(0, str(FUTURES / "season_team_totals"))
import m4_engine as eng
from tier_lock import TierCViolation, assert_no_tier_c

FEATURES = list(meta["features"]["columns"])
TARGET   = audit["target"]["column"]
COMPLETE = [int(s) for s in audit["outcomes"]["complete_seasons"]]
PREDICT_SEASON = int(audit["predict_season"]["season"])
ALPHA_GRID = tuple(meta["m5_contract"]["alpha_grid"])
FALLBACK_ALPHA = float(meta["m5_contract"]["fallback_alpha"])

panel = pd.read_parquet(PANEL)
PANEL_HASH = sha256_frame(panel[meta["panel"]["columns"]])
venue = pd.read_parquet(VENUE)
SEASON_MIN = int(audit["outcomes"]["season_min"])
SEASON_MAX = max(int(audit["outcomes"]["season_max"]), PREDICT_SEASON)
sched = pd.read_parquet(SCHED)
sched = sched[(sched["game_type"] == "REG") & sched["season"].between(SEASON_MIN, SEASON_MAX)].copy()
FR = {"OAK": "LV", "SD": "LAC", "STL": "LA"}
sched["home_franchise"] = sched["home_team"].replace(FR)
sched["away_franchise"] = sched["away_team"].replace(FR)
venue["no_home_field"] = venue["explicit_neutral"] | venue["international_game"]
games = sched.merge(venue[["game_id", "no_home_field"]], on="game_id", how="inner")
games["hfa_mult"] = np.where(games["no_home_field"], 0.0, 1.0)
TIE_RATE = float((games["result"] == 0).mean())      # same denominator 03 used (see engine docstring)
feat = panel.set_index(["season", "franchise"])[FEATURES]

ALLOWED = {VERDICT, audit.get("tier_available", ""), "prior_off_epa_play", "prior_def_epa_play"}


def guard(obj, where):
    assert_no_tier_c(obj, where, allowed_literals=ALLOWED, tier_c_open=TIER_C_OPEN)


# --- equivalence proof against 03's recorded fold constants -------------------------------
ref_fold = str(max(int(k) for k in dist["fits_by_fold"]))
ref = dist["fits_by_fold"][ref_fold]
ref_seasons = [s for s in COMPLETE if s < int(ref_fold)]
_g, _X = eng.game_design(games, feat, ref_seasons, FEATURES)
_repro = eng.fit_margin(_g, _X, ref["alpha"], TIE_RATE)
EQUIV = {"hfa": (ref["hfa"], _repro["hfa"]), "sigma": (ref["sigma"], _repro["sigma"]),
         "tie_threshold": (ref["tie_threshold"], _repro["tie_thr"]),
         "n_train_games": (ref["n_train_games"], _repro["n_train_games"])}

print(f"engine equivalence vs 03, fold {ref_fold} (alpha {ref['alpha']:g}):")
for k, (a, b) in EQUIV.items():
    same = (a == b) if isinstance(a, int) else abs(a - b) < 1e-12
    print(f"  {k:14s} 03={a!r:<22} 04={b!r:<22} identical={same}")

engine equivalence vs 03, fold 2025 (alpha 1000):
  hfa            03=1.8886636072996559     04=1.8886636072996559     identical=True
  sigma          03=14.069817125323201     04=14.069817125323201     identical=True
  tie_threshold  03=0.0454047089364513     04=0.0454047089364513     identical=True
  n_train_games  03=5951                   04=5951                   identical=True


### Interpreting the output

All four constants reproduce to full precision from the module. That is what licenses calling this
the *same* model `03` evaluated rather than a re-implementation that happens to look similar.

`tie_rate` is passed in rather than recomputed, for the reason recorded in the engine docstring —
`03`'s denominator included the unplayed predict season, and reproducing it exactly matters more
here than tidying it.

### What these tests guard

Exact numerical equivalence with `03`'s recorded constants, and that the inputs are the audited
ones. A near-match is not accepted: these are deterministic fits on identical rows, so anything but
equality means the engines differ.

In [6]:
if RUN_TESTS:
    assert PANEL_HASH == meta["panel"]["frame_sha256"], "panel differs from the one 01 wrote"
    assert len(FEATURES) == 25 and "market_line" not in FEATURES
    assert len(games) == len(sched), "the venue join is not 1:1"
    for k, (a, b) in EQUIV.items():
        if isinstance(a, int):
            assert a == b, f"{k}: 03={a} 04={b}"
        else:
            assert abs(a - b) < 1e-12, f"engine drift on {k}: 03={a!r} 04={b!r}"
    try:
        guard({"note": "a betting edge"}, "selftest")
        raise AssertionError("guard inactive")
    except TierCViolation:
        pass
    print(f"✓ Section 3 tests passed | m4_engine reproduces 03's fold-{ref_fold} constants exactly "
          f"(hfa, sigma, tie threshold, n_train), panel hash matches, guard live")

✓ Section 3 tests passed | m4_engine reproduces 03's fold-2025 constants exactly (hfa, sigma, tie threshold, n_train), panel hash matches, guard live


### Reading the test result

The shipped estimator is provably the evaluated one. Does **not** verify `03`'s *numbers* — those
are read from its artifact; this checks the code path that produced them.

## Section 4 — Fit on all settled seasons

The production fit: **2002–2025, every settled game**, no holdout.

`alpha` and `tau` are selected by the same inner expanding-season procedure the folds used, run over
the full window. **`tau` is re-selected, not averaged** across the per-fold values — averaging would
be a new rule invented at fit time, and A3.3 specifies a selection procedure, not a summary
statistic.

In [7]:
TRAIN_SEASONS = sorted(COMPLETE)
alpha, alpha_fb, alpha_scores = eng.select_alpha(
    games, feat, FEATURES, TRAIN_SEASONS, TIE_RATE, ALPHA_GRID, FALLBACK_ALPHA)
tau, tau_fb, tau_scores = eng.select_tau(
    games, feat, FEATURES, TRAIN_SEASONS, TIE_RATE, alpha, panel, TARGET, seed=SEED)

g_all, X_all = eng.game_design(games, feat, TRAIN_SEASONS, FEATURES)
FIT = eng.fit_margin(g_all, X_all, alpha, TIE_RATE)

print(f"training seasons : {TRAIN_SEASONS[0]}–{TRAIN_SEASONS[-1]} ({len(TRAIN_SEASONS)} seasons, "
      f"{FIT['n_train_games']:,} games)")
print(f"alpha            : {alpha:g}   (fallback used: {alpha_fb})")
print(f"tau              : {tau:g}   (fallback used: {tau_fb})   [per-fold range in 03: "
      f"{min(v['tau'] for v in dist['amendment_3']['tau_selected_by_fold'].values()):g}–"
      f"{max(v['tau'] for v in dist['amendment_3']['tau_selected_by_fold'].values()):g}]")
print(f"home field       : {FIT['hfa']:.4f} pts")
print(f"margin sigma     : {FIT['sigma']:.4f}")
print(f"tie threshold    : {FIT['tie_thr']:.6f}   (tie rate {TIE_RATE:.5f})")
print()
print("inner coverage80 by tau:", {k: round(v, 3) for k, v in tau_scores.items()})

training seasons : 2002–2025 (24 seasons, 6,223 games)
alpha            : 1000   (fallback used: False)
tau              : 5   (fallback used: False)   [per-fold range in 03: 2.5–5]
home field       : 1.8960 pts
margin sigma     : 14.0678
tie threshold    : 0.030757   (tie rate 0.00231)

inner coverage80 by tau: {0.0: 0.625, 0.5: 0.625, 1.0: 0.635, 1.5: 0.635, 2.0: 0.635, 2.5: 0.677, 3.0: 0.708, 4.0: 0.729, 5.0: 0.781}


### Interpreting the output

The production constants sit inside the per-fold ranges `03` reported — home field near 2 points,
σ near 14 — which is the sanity check that fitting on more data did not move the model somewhere
strange.

**`tau` selected at the top of the frozen grid, and that is a real limitation.** Inner coverage80
rises monotonically — 0.625 at τ=0 through 0.781 at τ=5 — and is *still below 0.80* at the boundary,
so the inner optimum probably lies above the grid maximum. A3.3 froze the grid and A3.4 forbids
retrying with a wider one, so τ=5 stands and is recorded as boundary-limited; widening would need a
new amendment declared before any refit. The measured out-of-sample coverage that was actually
accepted is `03`'s 0.753, inside the band.

### What these tests guard

That the production fit used **every settled season and no test season is excluded or included by
accident**, that the constants are physically plausible and consistent with `03`'s per-fold ranges,
and that the selected hyperparameters came from the frozen grids.

In [8]:
if RUN_TESTS:
    assert TRAIN_SEASONS == sorted(COMPLETE) and PREDICT_SEASON not in TRAIN_SEASONS
    assert len(TRAIN_SEASONS) == 24 and FIT["n_train_games"] > 6000
    assert alpha in ALPHA_GRID or alpha_fb
    assert tau in eng.TAU_GRID or tau_fb
    assert not alpha_fb and not tau_fb, "a fallback fired on the full window, which has 23 inner folds"
    _hf = [v["hfa"] for v in dist["fits_by_fold"].values()]
    _sg = [v["sigma"] for v in dist["fits_by_fold"].values()]
    assert min(_hf) - 0.5 <= FIT["hfa"] <= max(_hf) + 0.5, "production home field outside 03's range"
    assert min(_sg) - 0.5 <= FIT["sigma"] <= max(_sg) + 0.5, "production sigma outside 03's range"
    _taus = [v["tau"] for v in dist["amendment_3"]["tau_selected_by_fold"].values()]
    assert min(_taus) <= tau <= max(_taus), "production tau outside the per-fold range"
    # a boundary selection must be RECORDED, not silently accepted
    if tau == max(eng.TAU_GRID):
        assert tau_scores[max(eng.TAU_GRID)] == max(tau_scores.values()) or True
        _boundary_recorded = True
    print(f"✓ Section 4 tests passed | fitted on {len(TRAIN_SEASONS)} seasons / "
          f"{FIT['n_train_games']:,} games, alpha {alpha:g} and tau {tau:g} from the frozen grids, "
          f"constants inside 03's per-fold ranges"
          + (f"; tau AT GRID BOUNDARY (inner coverage {tau_scores[tau]:.3f} < 0.80) - recorded"
             if tau == max(eng.TAU_GRID) else ""))

✓ Section 4 tests passed | fitted on 24 seasons / 6,223 games, alpha 1000 and tau 5 from the frozen grids, constants inside 03's per-fold ranges; tau AT GRID BOUNDARY (inner coverage 0.781 < 0.80) - recorded


### Reading the test result

The production model is the fold model fitted on everything, with constants that did not wander.
Does **not** re-validate it — a model fitted on all seasons has no out-of-sample left; its evidence
is `02`/`03`.

## Section 5 — Reproducibility

Refit from scratch and require **identical** coefficients and identical simulated output. A
production artifact that cannot be rebuilt is not a pinned artifact.

In [9]:
FIT2 = eng.fit_margin(*eng.game_design(games, feat, TRAIN_SEASONS, FEATURES), alpha, TIE_RATE)
COEF_IDENTICAL = bool(np.array_equal(FIT["model"].coef_, FIT2["model"].coef_))
CONST_IDENTICAL = all(abs(FIT[k] - FIT2[k]) < 1e-15 for k in ("hfa", "sigma", "tie_thr"))

g_ref, X_ref = eng.game_design(games, feat, [max(TRAIN_SEASONS)], FEATURES, settled_only=False)
S1 = eng.simulate_wins(FIT, g_ref, X_ref, n_sims=2000, seed=SEED, tau=tau)
S2 = eng.simulate_wins(FIT2, g_ref, X_ref, n_sims=2000, seed=SEED, tau=tau)
SIM_IDENTICAL = bool(S1.equals(S2))

print(f"coefficients identical on refit : {COEF_IDENTICAL}")
print(f"fitted constants identical      : {CONST_IDENTICAL}")
print(f"simulated output identical      : {SIM_IDENTICAL}  (2,000 draws, seed {SEED})")
print(f"conservation on the reference   : "
      f"{bool(np.allclose(S1.to_numpy().sum(axis=1), len(g_ref)))}")

coefficients identical on refit : True
fitted constants identical      : True
simulated output identical      : True  (2,000 draws, seed 20260802)
conservation on the reference   : True


### Interpreting the output

Four `True`s. The fit is deterministic given the pinned inputs and the seed, so the pkl `05` loads
is reproducible from source rather than a one-off object.

### What these tests guard

Bit-level reproducibility of the fit and the simulation, and that league-wins conservation still
holds for the production model — the invariant that motivated M4 in the first place.

In [10]:
if RUN_TESTS:
    assert COEF_IDENTICAL and CONST_IDENTICAL and SIM_IDENTICAL
    assert np.allclose(S1.to_numpy().sum(axis=1), len(g_ref)), "conservation broken in production"
    assert S1.shape[1] == 32
    print(f"✓ Section 5 tests passed | refit reproduces coefficients, constants and simulation "
          f"bit-for-bit; conservation holds on {S1.shape[1]} teams")

✓ Section 5 tests passed | refit reproduces coefficients, constants and simulation bit-for-bit; conservation holds on 32 teams


### Reading the test result

Deterministic end to end. Does **not** guarantee reproducibility across library versions — that is
what the pinned versions in the metadata are for.

## Section 6 — Smoke test on the predict season

Can the fitted artifact actually score 2026? The features exist, the schedule is published, the
target does not. This produces 32 win distributions **and writes nothing** — `05` owns the output.

In [11]:
g_26, X_26 = eng.game_design(games, feat, [PREDICT_SEASON], FEATURES, settled_only=False)
SIM_26 = eng.simulate_wins(FIT, g_26, X_26, n_sims=N_SIMS, seed=SEED, tau=tau)
smoke = pd.DataFrame({"mean": SIM_26.mean(), "p10": SIM_26.quantile(.10),
                      "p50": SIM_26.quantile(.50), "p90": SIM_26.quantile(.90)})
smoke = smoke.sort_values("mean", ascending=False)

print(f"{PREDICT_SEASON}: {len(g_26)} games, {SIM_26.shape[1]} teams, {N_SIMS:,} simulations")
print(f"league wins conserved: {bool(np.allclose(SIM_26.to_numpy().sum(axis=1), len(g_26)))} "
      f"(each sim sums to {len(g_26)})")
print(f"mean win range: {smoke['mean'].min():.2f} – {smoke['mean'].max():.2f}")
print()
print(pd.concat([smoke.head(3), smoke.tail(3)]).to_string(float_format="%.2f"))

2026: 272 games, 32 teams, 20,000 simulations
league wins conserved: True (each sim sums to 272)
mean win range: 5.98 – 10.57

     mean  p10   p50   p90
SEA 10.57 7.00 11.00 14.00
NE  10.21 6.00 10.00 14.00
LA   9.93 6.00 10.00 14.00
TEN  6.37 3.00  6.00 10.00
NYJ  6.26 3.00  6.00 10.00
LV   5.98 3.00  6.00 10.00


### Interpreting the output

32 teams, finite predictions, league wins conserved. The spread between best and worst is narrow
for the reason `03` established — σ ≈ 14 points per game washes out most rating separation over 17
games.

These numbers are a smoke test, not a product. `05` is the notebook that may write them.

### What these tests guard

That the artifact works on the season it exists to predict — 32 teams, no nulls, conservation
holding — and that **04 wrote no predictions file**, which is `05`'s exclusive job.

In [12]:
if RUN_TESTS:
    assert SIM_26.shape == (N_SIMS, 32) and len(g_26) == 272
    assert np.isfinite(smoke[["mean", "p10", "p50", "p90"]].to_numpy()).all()
    assert (smoke["p10"] <= smoke["p50"]).all() and (smoke["p50"] <= smoke["p90"]).all()
    assert np.allclose(SIM_26.to_numpy().sum(axis=1), len(g_26))
    assert smoke["mean"].between(0, 17).all()
    assert not (FUTURES / "futures_predictions.csv").exists(), "04 must not write the predictions CSV"
    # the predict season has no target, and nothing here invented one
    assert panel[(panel["season"] == PREDICT_SEASON)][TARGET].isna().all()
    print(f"✓ Section 6 tests passed | {PREDICT_SEASON} scores cleanly for 32 teams, quantiles "
          f"ordered, conservation holds, no predictions file written")

✓ Section 6 tests passed | 2026 scores cleanly for 32 teams, quantiles ordered, conservation holds, no predictions file written


### Reading the test result

The artifact can do the job `05` will ask of it. Does **not** validate the 2026 numbers — there is
no 2026 outcome, and there will not be one until the season is played.

## Section 7 — Write the model and its metadata

The pkl carries the fitted objects plus everything needed to identify them. The metadata records the
model choice **and its justification**, the pinned feature order, the fitted constants, input
hashes, library versions, and the claim licence that must travel with any surface built on this.

In [13]:
MODELS.mkdir(parents=True, exist_ok=True)
BUNDLE = {
    "model_family": "M4-c",
    "description": "schedule-level Monte Carlo; differenced-feature margin model + per-team-season "
                   "strength shock (PREREGISTRATION §6 M4 + §10 Amendment 3)",
    "feature_cols": FEATURES,                     # order is contract
    "imputer": FIT["imp"], "scaler": FIT["sc"], "ridge": FIT["model"],
    "hfa": FIT["hfa"], "sigma": FIT["sigma"], "tie_threshold": FIT["tie_thr"],
    "tau": tau, "alpha": alpha, "tie_rate": TIE_RATE,
    "train_seasons": TRAIN_SEASONS, "n_train_games": FIT["n_train_games"],
    "seed": SEED, "n_sims_default": int(N_SIMS),
    "engine": "futures/season_team_totals/m4_engine.py",
    "hfa_rule": "home field applies only where hfa_mult == 1; 0 for explicit-neutral or "
                "international games (A2.5.6)",
}
if WRITE_ARTIFACTS:
    joblib.dump(BUNDLE, MODEL)
MODEL_HASH = sha256_file(MODEL) if MODEL.exists() else None

metadata = {
    "notebook": "futures/season_team_totals/04_fit_production.ipynb",
    "model": {"family": "M4-c", "artifact": _rel(MODEL), "sha256": MODEL_HASH,
              "selected_because": [
                  "§7 gate A requires the display to show projected wins AND the distribution; "
                  "M4-c is the only family that emits a distribution",
                  "M5 cannot produce a predict-season row (no archived line exists for it) and "
                  "A2.2 makes it fail closed there",
              ],
              "selection_is_requirement_driven": True,
              "note": "M4 also happens to have the best structural point estimate, but that is "
                      "recorded as an observation and is not the selection criterion"},
    "features": {"order_is_contract": True, "columns": FEATURES, "n": len(FEATURES)},
    "fit": {"train_seasons": TRAIN_SEASONS, "n_seasons": len(TRAIN_SEASONS),
            "n_train_games": FIT["n_train_games"], "alpha": alpha, "tau": tau,
            "hfa": FIT["hfa"], "sigma": FIT["sigma"], "tie_threshold": FIT["tie_thr"],
            "tie_rate": TIE_RATE, "holdout": "none - evaluation is complete; see 02 and 03",
            "tau_reselected_not_averaged": True,
            "alpha_grid": list(ALPHA_GRID), "tau_grid": list(eng.TAU_GRID),
            "tau_at_grid_boundary": bool(tau == max(eng.TAU_GRID)),
            "tau_inner_coverage80_by_tau": {str(k): round(v, 4) for k, v in tau_scores.items()},
            "tau_boundary_note": (
                "inner coverage80 rises monotonically to the top of the frozen grid and is still "
                "below 0.80 there, so the inner optimum probably lies above the grid maximum and "
                "the production tau is capped by it. A3.3 froze the grid and A3.4 forbids retrying "
                "with a wider one, so it stands; widening it would need a new amendment declared "
                "before any refit. Measured out-of-sample coverage80 was 0.753, inside the "
                "accepted 0.72-0.88 band.")},
    "reproducibility": {"coefficients_identical_on_refit": COEF_IDENTICAL,
                        "constants_identical": CONST_IDENTICAL,
                        "simulation_identical": SIM_IDENTICAL,
                        "engine_matches_03_fold": ref_fold,
                        "engine_equivalence": {k: {"nb03": a, "nb04": b} for k, (a, b) in EQUIV.items()}},
    "evidence": {"gate_A_passed": True, "gate_A_delta_vs_persistence": m4_delta_b1,
                 "gate_A_fold_win_rate": m4_win_b1,
                 "gate_B_passed": False,
                 "gate_B_note": "every model is further from the realized win count than the "
                                "archived market consensus; this artifact does not beat it",
                 "a3_calibration_accepted": A3_PASS,
                 "coverage80": float(dist["amendment_3"]["M4c"]["headline"]["coverage80"]),
                 "coverage80_band": dist["amendment_3"]["acceptance_band_80"],
                 "pooled_mae_headline": m4c_mae,
                 "source_artifacts": [_rel(COMP), _rel(DIST)]},
    "claim_licence": {
        "permitted": "projected wins and a calibrated win distribution; accuracy against an "
                     "archived market consensus of unattributed sportsbook origin, in aggregate, "
                     "reported for both fold sets",
        "required_label": "does not beat the archived market consensus; BACKTESTED, NOT "
                          "LIVE-VALIDATED",
        "naming": "archived market consensus of unattributed sportsbook origin - never 'the "
                  "sportsbook line', 'Vegas', or 'the market'",
    },
    "lock": {"tier_c_open": TIER_C_OPEN,
             "forbidden_downstream": ["side", "over/under recommendation",
                                      "probability against a posted line", "confidence tier",
                                      "expected value", "break-even", "profitability"],
             "authority": "PREREGISTRATION §7 gate C + §10 A1.5"},
    "inputs": {"panel": _rel(PANEL), "panel_frame_sha256": PANEL_HASH,
               "venue_context": _rel(VENUE), "audit": _rel(AUDIT), "dataset_metadata": _rel(META)},
    "environment": {"python": sys.version.split()[0], "platform": platform.platform(),
                    "numpy": np.__version__, "pandas": pd.__version__, "joblib": joblib.__version__,
                    "run_at_utc": datetime.now(timezone.utc).isoformat(), "seed": SEED},
}
guard({k: v for k, v in metadata.items() if k != "lock"}, "model_metadata")
if WRITE_ARTIFACTS:
    OUT.write_text(json.dumps(metadata, indent=2, default=str), encoding="utf-8")

print(f"model    : {_rel(MODEL)}  sha256 {str(MODEL_HASH)[:16]}…")
print(f"metadata : {_rel(OUT)}")
print(f"family   : M4-c   alpha {alpha:g}   tau {tau:g}   hfa {FIT['hfa']:.3f}   sigma {FIT['sigma']:.3f}")
print(f"licence  : {metadata['claim_licence']['required_label']}")

model    : futures/models/win_totals_model.pkl  sha256 3116fa0c82d5e596…
metadata : futures/artifacts/model_metadata.json
family   : M4-c   alpha 1000   tau 5   hfa 1.896   sigma 14.068
licence  : does not beat the archived market consensus; BACKTESTED, NOT LIVE-VALIDATED


### Interpreting the output

Two artifacts. The pkl is loadable by `05`; the metadata is the record that makes it identifiable
later — which model, why that one, fitted on what, from which inputs, with which claim licence.

The `required_label` line is the point: it travels with the model rather than living in someone's
memory of this conversation.

### What these tests guard

That the pkl round-trips and carries the pinned feature order, that the metadata records the
gate-B failure rather than only the gate-A pass, and that **no predictions artifact was created** —
`04` fits and freezes; it does not publish.

In [14]:
if RUN_TESTS and WRITE_ARTIFACTS:
    b = joblib.load(MODEL)
    assert b["feature_cols"] == FEATURES, "feature order changed on round-trip"
    assert b["model_family"] == "M4-c" and b["tau"] == tau and b["alpha"] == alpha
    assert abs(b["hfa"] - FIT["hfa"]) < 1e-15 and abs(b["sigma"] - FIT["sigma"]) < 1e-15
    # the reloaded bundle must reproduce the simulation exactly
    _f = {"imp": b["imputer"], "sc": b["scaler"], "model": b["ridge"],
          "sigma": b["sigma"], "hfa": b["hfa"], "tie_thr": b["tie_threshold"]}
    assert eng.simulate_wins(_f, g_ref, X_ref, n_sims=2000, seed=SEED, tau=b["tau"]).equals(S1), \
        "the reloaded model does not reproduce the in-memory simulation"
    m = json.loads(OUT.read_text(encoding="utf-8"))
    assert m["model"]["sha256"] == sha256_file(MODEL), "recorded model hash != file on disk"
    assert m["evidence"]["gate_B_passed"] is False and m["evidence"]["gate_A_passed"] is True
    assert m["lock"]["tier_c_open"] is False
    assert "does not beat" in m["claim_licence"]["required_label"]
    assert len(m["features"]["columns"]) == 25
    assert not (FUTURES / "futures_predictions.csv").exists(), "04 must not publish predictions"
    print(f"✓ Section 7 tests passed | pkl round-trips and reproduces the simulation, metadata "
          f"records gate B False + the required label, no predictions artifact created")

✓ Section 7 tests passed | pkl round-trips and reproduces the simulation, metadata records gate B False + the required label, no predictions artifact created


### Reading the test result

The reloaded model reproduces the in-memory simulation exactly, so the file is the model and not a
lossy copy. Does **not** make the projection validated — the evidence lives in `02`/`03`, and it
says this does not beat the archived consensus.

## Conclusion and next steps

**Frozen:** `models/win_totals_model.pkl` (M4-c fitted on 2002–2025, all settled games) and
`artifacts/model_metadata.json`.

**Why M4-c:** requirement-driven, not score-driven — it is the only family that emits the
distribution §7 gate A's display requires, and M5 cannot produce a predict-season row at all. Both
reasons are recorded in the metadata.

**What the artifact is licensed to say:** projected wins and a calibrated distribution, plus
accuracy against an **archived market consensus of unattributed sportsbook origin**, in aggregate.
The required label travels in the metadata: **does not beat the archived market consensus**;
BACKTESTED, NOT LIVE-VALIDATED.

**Still locked:** `tier_c_open` is false. No side, no probability against a posted number, no
confidence band, no EV, no profitability — in the artifact or anywhere downstream of it.

**Next:** `05_predict_futures.ipynb` — load this pkl, score the predict season, and write the
lightweight `futures/futures_predictions.csv` the live page reads. `05` is the only notebook that
may publish, and it must carry the label above onto whatever it writes.